In [1]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
# from transformers import  GptOssForCausalLM, AutoTokenizer
import torch
import math
import pandas as pd
import json
from tqdm import tqdm
import ast
import nltk
from nltk.corpus import stopwords
import re

In [2]:
odf = pd.read_csv("smallhub_0-0_sampled.csv")


In [3]:
# fdf = odf.head(1)
fdf = odf.copy()

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
input_text = "The quick brown fox jumps over the lazy dog."

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,

    device_map="auto"
)
model.eval()
    # dtype=torch.float32,

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [5]:
#stitching and liloing
irdf = pd.read_csv("../week15/gpu/interpretive_repertoires.csv")
combined_rows = []
for _, f_row in fdf.iterrows():
    for _, ir_row in irdf.iterrows():
        combined_row = {
            **f_row.to_dict(),      # all columns from fdf
            **ir_row.to_dict()      # all columns from irdf
        }
        combined_rows.append(combined_row)

cdf = pd.DataFrame(combined_rows)
cdf["attestation"] = cdf["likes"].combine_first(cdf["dislikes"])
cdf["input_text_w_ir"] = (
    "AUTHOR BIO: \n\n" +
    cdf["attestation"].astype(str) +
    "\n ARTICLE HEADLINE: \n" +
    cdf["human_clean_title"].astype(str) +
    "\n ARTICLE TEXT: \n" +
    cdf["human_clean_text"].astype(str)
)

cdf["input_text"] = (
    "\n ARTICLE HEADLINE: \n" +
    cdf["human_clean_title"].astype(str) +
    "\n ARTICLE TEXT: \n" +
    cdf["human_clean_text"].astype(str)
)

# cdf["input_text"] = cdf["text"]
cdf.drop(columns=['likes', 'dislikes', 'guidewords'])
funkyoutputblocker = None

In [6]:
logprobs = []
full_logprobs = []
for idx, row in tqdm(cdf.iterrows(), total=len(cdf)):
    inputs = tokenizer(row['input_text_w_ir'], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    input_ids = inputs["input_ids"]
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    average_log_prob = token_log_probs.mean().item()
    logprobs.append(average_log_prob)
    # full_logprobs.append(token_log_probs)

cdf["avg_logprob_with_attest"] = logprobs
# cdf["logprobs"] = full_logprobs

Token indices sequence length is longer than the specified maximum sequence length for this model (2238 > 2048). Running this sequence through the model will result in indexing errors
100%|██████████| 5992/5992 [32:55<00:00,  3.03it/s]  


In [7]:
# cdf.head()
cdf.to_csv("october_ir_1.csv")

In [8]:
logprobs = []
full_logprobs = []
for idx, row in tqdm(cdf.iterrows(), total=len(cdf)):
    inputs = tokenizer(row['input_text'], return_tensors="pt").to(model.device)
    # inputs = tokenizer(row['input_text'], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    input_ids = inputs["input_ids"]
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    average_log_prob = token_log_probs.mean().item()
    logprobs.append(average_log_prob)
    # full_logprobs.append(token_log_probs)

cdf["avg_logprob_just_text"] = logprobs

100%|██████████| 5992/5992 [25:22<00:00,  3.93it/s]  


In [9]:
cdf.to_csv("october_ir_2.csv")